In [40]:
import pandas as pd
import numpy as np

# Read the filtered CSV file
input_file = '/mnt/d/heatpump_data/u_values/germany_u_values_cleaned_filtered.csv'
df = pd.read_csv(input_file)

print(f"Total rows in input file: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Total rows in input file: 62
Columns: ['Code_Construction', 'Code_StatusDataset', 'Code_Country', 'Code_ElementType', 'Code_DataType_Construction', 'Code_Construction_ConstructionYearClass', 'Number_Construction_Variant', 'Type_Construction', 'Type_Construction_National', 'Description_Construction', 'Description_Construction_National', 'Remark_Construction', 'Year1_Construction', 'Year2_Construction', 'U']


,Code_Construction,Code_StatusDataset,Code_Country,Code_ElementType,Code_DataType_Construction,Code_Construction_ConstructionYearClass,Number_Construction_Variant,Type_Construction,Type_Construction_National,Description_Construction,Description_Construction_National,Remark_Construction,Year1_Construction,Year2_Construction,U
0,DE.Ceiling.ReEx.01.01,Typology,DE,Ceiling,ReEx,DE.01,1,wooden beam ceiling with visible beams,Holzbalkendecke mit sichtbaren Balken,"wooden beams, cavity filled with clay/straw","Holzbalken, Strohlehmwickel im Gefach",NaN,0,1918.0,1.0
1,DE.Ceiling.ReEx.03.01,Typology,DE,Ceiling,ReEx,DE.03,1,wooden beam ceiling,Holzbalkendecke,"wooden beams, cavity filled with clay, sand or...","Holzbalken, Blindboden, im Gefach: Lehmschlag,...",NaN,0,1968.0,0.8
2,DE.Ceiling.ReEx.04.01,Typology,DE,Ceiling,ReEx,DE.04,1,cavity blocks ceiling,"Rippendecke, Stahlsteindecke, Gitterträgerdecke","cavity elements, reinforcement, concrete filli...","Stahlstein- oder Gitterträgerdecke, Bewehrung,...",NaN,1919,1968.0,2.1
3,DE.Ceiling.ReEx.06.01,Typology,DE,Ceiling,ReEx,DE.06,1,concrete ceiling with 5 cm insulation,Betondecke mit 5 cm Dämmung,"reinforced concrete, 5 cm insulation, cement s...","Stahlbeton, oberseitig 5 cm Dämmung, Zementest...",NaN,1958,1978.0,0.6
4,DE.Ceiling.ReEx.07.01,Typology,DE,Ceiling,ReEx,DE.07,1,concrete ceiling with 6 cm insulation,Betondecke mit 6 cm Dämmung,"reinforced concrete, 6 cm insulation, cement s...","Stahlbeton, oberseitig 6 cm Dämmung, Zementest...",NaN,1979,1983.0,0.5


In [41]:
# Parse Code_Construction to extract components
# Format: Country.Element.Renovation.Year_range.variant
def parse_code_construction(code):
    """Parse Code_Construction to extract all components.
    
    Args:
        code: Code_Construction string
        
    Returns:
        tuple: (Country, Element, Renovation, Year_range, Variant)
    """
    parts = code.split('.')
    if len(parts) >= 5:
        return (parts[0], parts[1], parts[2], parts[3], parts[4])
    return (None, None, None, None, None)

# Create component columns
df[['Country', 'Element', 'Renovation', 'Year_range', 'Variant']] = df['Code_Construction'].apply(
    lambda x: pd.Series(parse_code_construction(x))
)

# Convert Year_range to integer for comparison
df['Year_range_int'] = df['Year_range'].astype(int)

print("Sample parsed data:")
print(df[['Code_Construction', 'Country', 'Element', 'Renovation', 'Year_range', 'Variant', 'U']].head(10))

Sample parsed data:
       Code_Construction Country  Element Renovation Year_range Variant     U
0  DE.Ceiling.ReEx.01.01      DE  Ceiling       ReEx         01      01  1.00
1  DE.Ceiling.ReEx.03.01      DE  Ceiling       ReEx         03      01  0.80
2  DE.Ceiling.ReEx.04.01      DE  Ceiling       ReEx         04      01  2.10
3  DE.Ceiling.ReEx.06.01      DE  Ceiling       ReEx         06      01  0.60
4  DE.Ceiling.ReEx.07.01      DE  Ceiling       ReEx         07      01  0.50
5  DE.Ceiling.ReEx.08.01      DE  Ceiling       ReEx         08      01  0.40
6  DE.Ceiling.ReEx.09.01      DE  Ceiling       ReEx         09      01  0.35
7  DE.Ceiling.ReEx.10.01      DE  Ceiling       ReEx         10      01  0.30
8  DE.Ceiling.ReEx.11.01      DE  Ceiling       ReEx         11      01  0.25
9     DE.Door.ReEx.01.01      DE     Door       ReEx         01      01  3.00


In [42]:
# Find missing year ranges for each Country.Element.Renovation.Variant combination
# For each missing year range, find the closest previous year range

def find_closest_previous_year_range(missing_year, existing_years):
    """Find the closest previous (lower) year range.
    
    Args:
        missing_year: The missing year range as integer
        existing_years: List of existing year ranges as integers
        
    Returns:
        int: Closest previous year range, or None if no previous exists
    """
    previous_years = [y for y in existing_years if y < missing_year]
    if previous_years:
        return max(previous_years)
    return None

# Group by Country, Element, Renovation, Variant
new_rows = []

for (country, element, renovation, variant), group in df.groupby(['Country', 'Element', 'Renovation', 'Variant']):
    existing_years = sorted(group['Year_range_int'].unique())
    min_year = min(existing_years)
    max_year = max(existing_years)
    
    # Find all possible year ranges between min and max
    all_possible_years = set(range(min_year, max_year + 1))
    missing_years = sorted(all_possible_years - set(existing_years))
    
    if missing_years:
        print(f"\n{country}.{element}.{renovation}.XX.{variant}:")
        print(f"  Existing: {existing_years}")
        print(f"  Missing: {missing_years}")
        
        # For each missing year, find closest previous and create new row
        for missing_year in missing_years:
            closest_prev = find_closest_previous_year_range(missing_year, existing_years)
            if closest_prev is not None:
                # Get the row with the closest previous year range
                source_row = group[group['Year_range_int'] == closest_prev].iloc[0].copy()
                
                # Update the year range fields
                source_row['Year_range'] = f"{missing_year:02d}"
                source_row['Year_range_int'] = missing_year
                source_row['Code_Construction'] = f"{country}.{element}.{renovation}.{source_row['Year_range']}.{variant}"
                source_row['Code_Construction_ConstructionYearClass'] = f"{country}.{source_row['Year_range']}"
                
                # Clear description fields (as requested)
                source_row['Description_Construction'] = ''
                source_row['Description_Construction_National'] = ''
                source_row['Type_Construction'] = ''
                source_row['Type_Construction_National'] = ''
                
                new_rows.append(source_row)
                print(f"    {missing_year:02d} -> copied from {closest_prev:02d}")

print(f"\nTotal new rows to add: {len(new_rows)}")

Year range mapping from reference file:
  00: 0 - 0.0
  01: 0 - 1859.0
  02: 1860 - 1918.0
  03: 1919 - 1948.0
  04: 1949 - 1957.0
  05: 1958 - 1968.0
  06: 1969 - 1978.0
  07: 1979 - 1983.0
  08: 1984 - 1994.0
  09: 1995 - 2001.0
  10: 2002 - 2009.0
  11: 2010 - 2015.0
  12: 2016 - 9999.0

Updating existing rows with correct year ranges...
Updated 62 existing rows with correct year ranges from reference file.

DE.Ceiling.ReEx.XX.01:
  Existing: [np.int64(1), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
  Missing: [2, 5]
    02 -> copied from 01
    05 -> copied from 04

DE.Door.ReEx.XX.01:
  Existing: [np.int64(1), np.int64(3), np.int64(9), np.int64(11)]
  Missing: [2, 4, 5, 6, 7, 8, 10]
    02 -> copied from 01
    04 -> copied from 03
    05 -> copied from 03
    06 -> copied from 03
    07 -> copied from 03
    08 -> copied from 03
    10 -> copied from 09

DE.Floor.ReEx.XX.02:
  Existing: [np.int64(1), np.int64(3)]
  Mis

In [43]:
# Combine original dataframe with new rows
if new_rows:
    new_df = pd.DataFrame(new_rows)
    # Remove temporary columns before combining
    df_final = pd.concat([df.drop(columns=['Year_range_int']), 
                          new_df.drop(columns=['Year_range_int'])], 
                         ignore_index=True)
else:
    df_final = df.drop(columns=['Year_range_int'])

# Sort by Code_Construction for better readability
df_final = df_final.sort_values('Code_Construction').reset_index(drop=True)

print(f"Original rows: {len(df)}")
print(f"New rows added: {len(new_rows)}")
print(f"Total rows in output: {len(df_final)}")

Original rows: 62
New rows added: 50
Total rows in output: 112


In [44]:
# Verify the filling worked - check a specific example
print("\nVerification - Checking DE.Roof.SyAv.XX.03:")
roof_syav = df_final[df_final['Code_Construction'].str.startswith('DE.Roof.SyAv.') & 
                     df_final['Code_Construction'].str.endswith('.03')]
print(roof_syav[['Code_Construction', 'Year_range', 'U']].to_string(index=False))


Verification - Checking DE.Roof.SyAv.XX.03:
 Code_Construction Year_range    U
DE.Roof.SyAv.01.03         01 1.09
DE.Roof.SyAv.02.03         02 1.09
DE.Roof.SyAv.03.03         03 1.09
DE.Roof.SyAv.04.03         04 1.09
DE.Roof.SyAv.05.03         05 1.09
DE.Roof.SyAv.06.03         06 1.09
DE.Roof.SyAv.07.03         07 0.45
DE.Roof.SyAv.08.03         08 0.45
DE.Roof.SyAv.09.03         09 0.34


In [45]:
# Remove temporary grouping columns before saving
df_final = df_final.drop(columns=['Country', 'Element', 'Renovation', 'Variant'])

# Save the filled dataframe
output_file = '/mnt/d/heatpump_data/u_values/germany_u_values_cleaned_filled.csv'
df_final.to_csv(output_file, index=False)

print(f"\nFilled CSV saved to: {output_file}")
print(f"Total rows saved: {len(df_final)}")


Filled CSV saved to: /mnt/d/heatpump_data/u_values/germany_u_values_cleaned_filled.csv
Total rows saved: 112
